# Modern Face Detection and Blurring with MediaPipe Tasks

This notebook uses the MediaPipe Tasks API to detect faces in images, videos, or live webcam feeds and applies a Gaussian blur to protect privacy.

Requirements: Python 3.12, mediapipe, opencv-python, numpy.

Required File: detector.tflite must be in the project directory.

#  Libraries 

In [10]:
import os
import argparse
import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# 1. Core Processing Function

In [11]:
def process_img(img, detector):
    """
    Detects faces in a frame and applies a blur effect.
    """
    # Convert OpenCV BGR to MediaPipe RGB
    rgb_frame = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    # Perform detection
    detection_result = detector.detect(mp_image)

    if detection_result.detections:
        for detection in detection_result.detections:
            # Get bounding box coordinates
            bbox = detection.bounding_box
            x, y, w, h = bbox.origin_x, bbox.origin_y, bbox.width, bbox.height

            # Ensure coordinates are within image boundaries
            x, y = max(0, x), max(0, y)
            
            # Apply blur to the region of interest (ROI)
            face_roi = img[y:y+h, x:x+w]
            if face_roi.size > 0:
                blurred_face = cv2.blur(face_roi, (50, 50))
                img[y:y+h, x:x+w] = blurred_face

    return img

# 2- Initialization & Directory Setup

In [12]:
# Setup output directory
output_dir = './output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Initialize MediaPipe Tasks Face Detector
model_path = 'detector.tflite' 

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.FaceDetectorOptions(base_options=base_options)
detector = vision.FaceDetector.create_from_options(options)

print("Detector initialized successfully.")

Detector initialized successfully.


# 3- Input Configuration  & Execution Logic

In [8]:
# USER SETTINGS
MODE = 'webcam'  # Options: 'image', 'video', 'webcam'
FILE_PATH = 'path/to/your/file.jpg' # Only for image/video mode

try:
    if MODE == "image":
        img = cv2.imread(FILE_PATH)
        if img is not None:
            img = process_img(img, detector)
            cv2.imwrite(os.path.join(output_dir, 'output.png'), img)
            print("Image processed and saved.")

    elif MODE == 'video':
        cap = cv2.VideoCapture(FILE_PATH)
        ret, frame = cap.read()
        if ret:
            h, w, _ = frame.shape
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(os.path.join(output_dir, 'output.mp4'), fourcc, 25, (w, h))
            
            while ret:
                frame = process_img(frame, detector)
                out.write(frame)
                ret, frame = cap.read()
            cap.release()
            out.release()
            print("Video saved.")

    elif MODE == 'webcam':
        cap = cv2.VideoCapture(0) 
        print("Webcam started. Press 'q' in the window to quit.")
        while True:
            ret, frame = cap.read()
            if not ret: break
            
            frame = process_img(frame, detector)
            cv2.imshow('Modern Face Blur', frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        cap.release()
        cv2.destroyAllWindows()

except Exception as e:
    print(f"An error occurred: {e}")

Webcam started. Press 'q' in the window to quit.


# 4- Cleanup

In [9]:
detector.close()
print("Detector closed and resources freed.")

Detector closed and resources freed.
